# OASIS-2 Multimodal Training Pipeline

This notebook is the entry point for the **Multimodal Alzheimer's AI Monitoring Platform**.
It fuses 3D brain MRI scans with clinical metadata (Age, Sex, MMSE) using the reliability-focused multimodal ResNet pipeline.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Constants for your environment
DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_multimodal_v2' # Reliability-focused multimodal candidate

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(f"{name}: {'[OK]' if path.exists() else '[MISSING]'} {path}")

Mounted at /content/drive
DRIVE_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud/backend_runtime


In [2]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = '95132f8' # Latest optimized commit

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

# Clean up any stale repo clones
for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print(f'Active commit: {repo_commit}')

# Hotfix for older GitHub clone: oasis2_research imports the pre-rename private helper.
oasis2_research_path = BACKEND_ROOT / 'src' / 'training' / 'oasis2_research.py'
oasis2_research_text = oasis2_research_path.read_text(encoding='utf-8')
oasis2_research_text = oasis2_research_text.replace('_resolve_loss_class_weights,', 'resolve_loss_class_weights,')
oasis2_research_path.write_text(oasis2_research_text, encoding='utf-8')
print('Patched oasis2_research.py loss class-weight import')

RUNNING git-clone: git clone https://github.com/Billrichard209/cerebrasense.git /content/cerebrasense
Cloning into '/content/cerebrasense'...

RUNNING git-checkout-main: git checkout main
Your branch is up to date with 'origin/main'.

Already on 'main'

RUNNING pip-install: python3 -m pip install -r /content/cerebrasense/alz_backend/requirements-colab.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 94.9 MB/s eta 0:00:00

RUNNING git-rev-parse: git rev-parse HEAD
a045d87162c4a93ea2776faf5eda4e40a42abaaf

Active commit: a045d87162c4a93ea2776faf5eda4e40a42abaaf


### Start Multimodal Training
Initiates a 50-epoch training run fusing MRI volumes with clinical metadata (Age, Sex, MMSE).

In [5]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Change runtime type to T4 GPU before proceeding.")

os.chdir(BACKEND_ROOT)

!python scripts/train_oasis2_colab.py \
    --project-root {BACKEND_ROOT} \
    --runtime-root {RUNTIME_ROOT} \
    --bundle-root {OASIS2_BUNDLE_ROOT} \
    --run-name {RUN_NAME} \
    --epochs 50 \
    --batch-size 2 \
    --gradient-accumulation-steps 4 \
    --num-workers 2 \
    --image-size 96 96 96 \
    --learning-rate 1e-4 \
    --weight-decay 0.01 \
    --scheduler cosine \
    --loss focal_loss \
    --class-weights 1.05 1.0 \
    --focal-gamma 1.0 \
    --temporal-lambda 0.15 \
    --weighted-sampling \
    --seed 42 \
    --split-seed 42 \
    --device auto \
    --config configs/oasis2_train_multimodal_v2.yaml

Traceback (most recent call last):
  File "/content/cerebrasense/alz_backend/scripts/train_oasis2_colab.py", line 27, in <module>
    from scripts.train_oasis2 import apply_cli_overrides, build_parser as build_train_parser  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cerebrasense/alz_backend/scripts/train_oasis2.py", line 14, in <module>
    from src.training.oasis2_research import default_oasis2_train_config_path, run_research_oasis2_training  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cerebrasense/alz_backend/src/training/oasis2_research.py", line 22, in <module>
    from .oasis_research import (
ImportError: cannot import name '_resolve_loss_class_weights' from 'src.training.oasis_research' (/content/cerebrasense/alz_backend/src/training/oasis_research.py). Did you mean: 'resolve_loss_class_weights'?


### Evaluate, Calibrate, and Export Run
Runs held-out OASIS-2 evaluation on Colab GPU, writes leaderboard/productization artifacts, and zips the evaluated run folder for local import.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

os.chdir(BACKEND_ROOT)
os.environ["ALZ_DATA_ROOT"] = str(RUNTIME_ROOT / "data")
os.environ["ALZ_OUTPUTS_ROOT"] = str(RUNTIME_ROOT / "outputs")
os.environ["ALZ_WORKSPACE_ROOT"] = str(DRIVE_ROOT)
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"ALZ_DATA_ROOT={os.environ['ALZ_DATA_ROOT']}")
print(f"ALZ_OUTPUTS_ROOT={os.environ['ALZ_OUTPUTS_ROOT']}")

run_root = RUNTIME_ROOT / "outputs" / "runs" / "oasis2" / RUN_NAME
checkpoint_path = run_root / "checkpoints" / "best_model.pt"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")


def run_live(cmd, *, cwd=None, label=None, allow_fail=False):
    print(f"\nRUNNING {label or cmd[0]}: {' '.join(map(str, cmd))}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True)
    print(f"RETURN_CODE {label or cmd[0]}: {completed.returncode}", flush=True)
    if completed.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(map(str, cmd))}")
    return completed


def patch_exact(path: Path, transform, *, label: str) -> None:
    text = path.read_text(encoding="utf-8")
    new_text = transform(text)
    if new_text != text:
        path.write_text(new_text, encoding="utf-8")
        print(f"Patched {label}: {path}")
    else:
        print(f"Patch already present for {label}: {path}")


patch_exact(
    BACKEND_ROOT / "src" / "training" / "oasis2_research.py",
    lambda text: text.replace("_resolve_loss_class_weights,", "resolve_loss_class_weights,"),
    label="oasis2 loss class-weight import",
)

oasis2_run_path = BACKEND_ROOT / "src" / "evaluation" / "oasis2_run.py"


def fix_oasis2_run_import(text: str) -> str:
    while "replace, replace" in text:
        text = text.replace("replace, replace", "replace")
    return text.replace(
        "from dataclasses import asdict, dataclass\n",
        "from dataclasses import asdict, dataclass, replace\n",
    )


patch_exact(oasis2_run_path, fix_oasis2_run_import, label="oasis2 evaluator replace import")

text = oasis2_run_path.read_text(encoding="utf-8")
needle = (
    "    checkpoint = load_oasis_checkpoint(resolve_oasis2_checkpoint_path(cfg, settings=settings), device=cfg.device)\n"
    "    model = build_model(model_cfg)\n"
)
insert = (
    "    if cfg.model_config_path is None:\n"
    "        resolved_settings = settings or get_app_settings()\n"
    "        resolved_config_path = resolve_oasis2_run_root(resolved_settings, cfg.run_name) / \"configs\" / \"resolved_config.json\"\n"
    "        if resolved_config_path.exists():\n"
    "            resolved_payload = json.loads(resolved_config_path.read_text(encoding=\"utf-8\"))\n"
    "            architecture = (\n"
    "                resolved_payload.get(\"training\", {}).get(\"model\", {}).get(\"architecture\")\n"
    "                or resolved_payload.get(\"model\", {}).get(\"architecture\")\n"
    "            )\n"
    "            if architecture:\n"
    "                model_cfg = replace(model_cfg, architecture=str(architecture))\n"
    "    checkpoint = load_oasis_checkpoint(resolve_oasis2_checkpoint_path(cfg, settings=settings), device=cfg.device)\n"
    "    model = build_model(model_cfg)\n"
)
if insert not in text:
    if needle not in text:
        raise RuntimeError("Could not patch oasis2_run.py architecture dispatch")
    oasis2_run_path.write_text(text.replace(needle, insert), encoding="utf-8")
    print("Patched oasis2 evaluator architecture dispatch")
else:
    print("Architecture dispatch patch already present")

patch_exact(
    BACKEND_ROOT / "src" / "models" / "multimodal.py",
    lambda text: text.replace("pretrained=True,", "pretrained=False,"),
    label="multimodal ResNet no-download eval build",
)

eval_path = BACKEND_ROOT / "src" / "evaluation" / "evaluate_oasis.py"


def fix_eval_dispatch(text: str) -> str:
    old = (
        "            logits = model(images)\n"
        "            probabilities = torch.softmax(logits, dim=1)\n"
    )
    new = (
        "            clinical = batch.get(\"clinical\")\n"
        "            if clinical is not None and hasattr(model, \"tabular_mlp\"):\n"
        "                logits = model(images, clinical.to(device))\n"
        "            else:\n"
        "                logits = model(images)\n"
        "            probabilities = torch.softmax(logits, dim=1)\n"
    )
    return text.replace(old, new)


patch_exact(eval_path, fix_eval_dispatch, label="multimodal eval clinical dispatch")

eval_model_config = BACKEND_ROOT / "configs" / "oasis2_multimodal_eval.yaml"
eval_model_config.write_text(
    "dataset: oasis1\n"
    "task: binary_3d_mri_classification\n"
    "architecture: resnet50_multimodal\n"
    "class_names: [nondemented, demented]\n"
    "expected_input_shape: [1, 1, 96, 96, 96]\n"
    "densenet:\n"
    "  name: densenet121\n"
    "  spatial_dims: 3\n"
    "  in_channels: 1\n"
    "  out_channels: 2\n"
    "  dropout_prob: 0.3\n"
    "embeddings:\n"
    "  enabled: false\n"
    "  hook_module_name: class_layers.flatten\n",
    encoding="utf-8",
)
print(f"Wrote eval model config: {eval_model_config}")

base_eval_cmd = [
    "python", "-u", "scripts/evaluate_oasis2_candidate.py",
    "--run-name", RUN_NAME,
    "--checkpoint-path", str(checkpoint_path),
    "--model-config-path", str(eval_model_config),
    "--device", "cuda",
    "--selection-metric", "balanced_accuracy",
    "--batch-size", "1",
    "--num-workers", "0",
    "--cache-rate", "0.0",
    "--image-size", "96", "96", "96",
]

run_live(base_eval_cmd + ["--max-batches", "2"], cwd=BACKEND_ROOT, label="smoke-eval-2-batches")
run_live(base_eval_cmd, cwd=BACKEND_ROOT, label="full-evaluate-calibrate-oasis2")

predictions_csv = run_root / "evaluation" / "post_train_test_best_model" / "predictions.csv"
audit_json = RUNTIME_ROOT / "outputs" / "reports" / "longitudinal" / f"audit_{RUN_NAME}.json"
if predictions_csv.exists():
    run_live([
        "python", "-u", "scripts/audit_temporal_paradoxes.py",
        "--predictions-csv", str(predictions_csv),
        "--output-json", str(audit_json),
    ], cwd=BACKEND_ROOT, label="temporal-paradox-audit")
else:
    print(f"Skipping temporal audit; missing {predictions_csv}")

run_live(["python", "-u", "scripts/analyze_oasis2_mixed_label_errors.py", "--run-name", RUN_NAME], cwd=BACKEND_ROOT, label="mixed-label-analysis", allow_fail=True)
run_live(["python", "-u", "scripts/build_oasis2_leaderboard.py", "--workspace-root", str(DRIVE_ROOT)], cwd=BACKEND_ROOT, label="leaderboard")
run_live(["python", "-u", "scripts/check_oasis2_productization.py", "--expected-run-name", RUN_NAME], cwd=BACKEND_ROOT, label="productization", allow_fail=True)

zip_base = DRIVE_ROOT / f"{RUN_NAME}_evaluated"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", run_root))
print(f"EVALUATED_RUN_ZIP={zip_path}")

metrics_path = run_root / "evaluation" / "post_train_test_best_model_threshold_balanced_accuracy" / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("\nFinal calibrated test metrics:")
    for key in ["auroc", "accuracy", "balanced_accuracy", "f1", "sensitivity", "specificity", "review_required_count"]:
        print(f"- {key}: {metrics.get(key)}")
else:
    print(f"Calibrated metrics not found: {metrics_path}")


### Result Summary
View final metrics after training completes.

In [4]:
import json
import pandas as pd
from pathlib import Path

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
summary_path = run_root / 'reports' / 'colab_run_summary.json'
metrics_path = run_root / 'metrics' / 'epoch_metrics.csv'

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f"\nTraining Summary for {RUN_NAME}:")
    print(f"- Best Epoch: {summary.get('best_epoch')}")
    print(f"- Best Val AUROC: {summary.get('best_monitor_value'):.4f}")
    print(f"- Best Checkpoint: {summary.get('best_checkpoint')}")

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print("\nLast 5 Epochs:")
    print(df.tail())



Training Summary for oasis2_multimodal_v1:
- Best Epoch: 13
- Best Val AUROC: 0.7302
- Best Checkpoint: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_multimodal_v1/checkpoints/best_model.pt

Last 5 Epochs:
    epoch  learning_rate  train_loss  val_loss  accuracy     auroc  precision  \
16     17       0.000045    0.289033  0.382969  0.574074  0.565158   0.566667   
17     18       0.000040    0.328256  0.455121  0.500000  0.483539   0.500000   
18     19       0.000035    0.337986  0.350443  0.537037  0.576132   0.545455   
19     20       0.000030    0.308145  0.388313  0.592593  0.594650   0.567568   
20     21       0.000025    0.296047  0.344450  0.629630  0.669410   0.620690   

      recall        f1  sensitivity  specificity  train_batches  val_batches  
16  0.629630  0.596491     0.629630     0.518519            130           27  
17  0.962963  0.658228     0.962963     0.037037            130           27  
18  0.444444  0.489796     0.44